# SPI Gorgon

### Read Data

In [1]:
import pandas as pd
df_spi=pd.read_excel("CVX_MTL_SPI_Gor.xlsx")

### Columns to keep

In [3]:
columns_to_keep=['CAT', 'CMPNT_ID','CMPNT_NAME', 'TAG_NUMBER', 'DISCIPLINE','STATUS', 'PROJECT_NO']
df_spi=df_spi[columns_to_keep]
rename_map={'CAT':'category', 'CMPNT_ID':'componentId','CMPNT_NAME':'componentName',  'TAG_NUMBER':'tagNumber', 'DISCIPLINE': 'discipline','STATUS':'status', 'PROJECT_NO': 'projectNumber'}
df_spi=df_spi.rename(columns=rename_map)
df_spi.to_csv('SPI_CVX_MTL.tsv', sep="\t", index=False)

#### Clean projectNumber column  and extract MocId

In [20]:
import pandas as pd
import re

def extract_mocs(value):
    """
    Extract and clean MOC numbers from the Project_no field.
    Returns a list of clean MOC values like 'MOC564805'.
    """

    if pd.isna(value):
        return []

    text = str(value)

    # Normalize separators
    text = text.replace("&", ",")
    text = text.replace(";", ",")
    text = text.replace("/", ",")
    text = text.replace(" and ", ",")
    text = text.replace("AND", ",")
    
    # Split multiple entries
    parts = re.split(r"[,\n]+", text)

    cleaned = []

    for p in parts:
        p = p.strip()

        # Find patterns with numbers and "MOC"
        match = re.search(r"(?:T?PMOC|TMOC|PMOC|MOC)[\s\-_]*([A-Za-z0-9]+)", p, re.IGNORECASE)

        if match:
            num = match.group(1).upper()
            cleaned.append("MOC" + num)
        else:
            # Catch patterns like "Rev 9 - MOC382271"
            match2 = re.search(r"MOC[\s\-_]*([A-Za-z0-9]+)", p, re.IGNORECASE)
            if match2:
                num = match2.group(1).upper()
                cleaned.append("MOC" + num)

    return cleaned


def clean_file(df_in: pd.DataFrame) -> pd.DataFrame:

    # Work on a copy to avoid mutating the caller's dataframe
    df = df_in.copy()

    # Create cleanMOC list column
    df["cleanMOC_list"] = df["projectNumber"].apply(extract_mocs)

    # Explode to multiple rows if more than one MOC
    df = df.explode("cleanMOC_list", ignore_index=True)

    # Rename final column
    df = df.rename(columns={"cleanMOC_list": "cleanMOC"})

    return df




#### Load Moc node and Tag node from SPI

In [23]:
import pandas as pd
import numpy as np
from neo4j import GraphDatabase

# =========================
# Connection settings
# =========================

URI = os.getenv("NEO4J_URI")
USERNAME = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")
DATABASE = os.getenv("NEO4J_DATABASE")

# =========================
# Constants
# =========================
#TSV_PATH = "spi_cleaned.tsv"  # <-- change if needed
SOR_NAME = "spi_gorgon"
NULL_TOKENS = {"", "-", "--", "---", "unset", "*"}

# =========================
# Data prep (clean in Python)
# =========================

def prepare_spi_rows(df_in: pd.DataFrame) -> pd.DataFrame:
#def prepare_spi_rows(tsv_path: str):
    #df = pd.read_csv(tsv_path, sep="\t", dtype=str).fillna("")
    df=df_in.copy()
    # Trim whitespace
    df = df.applymap(lambda v: v.strip() if isinstance(v, str) else v)

    # Convert null-like tokens to None
    df = df.applymap(lambda v: None if (isinstance(v, str) and v in NULL_TOKENS) else v)

    # Convert real NaN to None
    df = df.where(pd.notna(df), None)

    rows = []
    for r in df.to_dict(orient="records"):
        mocId = r.get("cleanMOC")
        tag_num = r.get("tagNumber")

        # Only process rows that have both MOC id and tag_number
        if not mocId or not tag_num:
            continue

        # Exclude matching keys and source_file so t += row won't overwrite it
        props = {
            k: v for k, v in r.items()
            if k not in ["cleanMOC", "tagNumber", "sourceFile"] and v is not None
        }
        props["cleanMOC"] = mocId
        props["tagNumber"] = tag_num

        rows.append(props)

    print(f"Prepared {len(rows)} rows for upload.")
    return rows

# =========================
# Cypher (no MOC creation; Tag MERGE; append source_file; relate if MOC exists)
# =========================
UPSERT_CYPHER = """

UNWIND $rows AS row


MATCH (m:Moc {mocId: row.cleanMOC})
SET m.sourceFile =
  CASE
    WHEN m.sourceFile IS NULL THEN ['spi']
    WHEN 'spi' IN m.sourceFile THEN m.sourceFile
    ELSE m.sourceFile + ['spi']
  END

// Clean tag number
WITH row, m,
CASE
  WHEN row.tagNumber IS NOT NULL
       AND row.tagNumber = row.tagNumber              // exclude NaN
       AND NOT toString(row.tagNumber) IN ["", "-", "--", "---", "unset", "*"]
  THEN row.tagNumber
  ELSE NULL
END AS tagVal

// Create/match Tag ONLY when tagVal is valid
FOREACH (_ IN CASE WHEN tagVal IS NULL THEN [] ELSE [1] END |
  MERGE (t:Tag {tagNumber: tagVal})
  SET
    // only allowed properties on Tag
    t.discipline = coalesce(row.discipline, t.discipline),
    // normalize (note: backslash must be escaped as \\\\ in Python, \\ in Cypher)
    t.tagNorm   = replace(replace(replace(replace(toLower(tagVal), "-", ""), "\\\\", ""), " ", ""), "f", ""),
    t.tagTokens = [tok IN split(tagVal, "-") WHERE trim(tok) <> "" | toLower(trim(tok))],
    // append-only source_file (no overwrite, no duplicates)
    t.sourceFile =
      CASE
        WHEN t.sourceFile IS NULL THEN ['spi_gorgon']
        WHEN 'spi_gorgon' IN t.sourceFile THEN t.sourceFile
        ELSE t.sourceFile + ['spi_gorgon']
      END, 
      t.siteName="Gorgon"
      
  MERGE (m)-[:ASSOCIATED_WITH_TAG]->(t)
)
"""





# =========================
# Run
# =========================
if __name__ == "__main__":
    

    rows = prepare_spi_rows(df_spi)

    driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
    try:
        with driver.session(database=DATABASE) as session:
            session.run(UPSERT_CYPHER, rows=rows, sor=SOR_NAME).consume()
        print("✅ Ingestion completed.")
    finally:
        driver.close()

C:\Users\rasma\AppData\Local\Temp\ipykernel_6756\105783983.py:35: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda v: v.strip() if isinstance(v, str) else v)
C:\Users\rasma\AppData\Local\Temp\ipykernel_6756\105783983.py:38: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda v: None if (isinstance(v, str) and v in NULL_TOKENS) else v)


Prepared 3235 rows for upload.
✅ Ingestion completed.


### Load spiRecord node

In [5]:
import pandas as pd
from neo4j import GraphDatabase
import numpy as np
import pandas as pd
from neo4j import GraphDatabase


URI = os.getenv("NEO4J_URI")
USERNAME = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")
DATABASE = os.getenv("NEO4J_DATABASE")

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))


# ======================================================
# 1. Ensure INDEXES — CRITICAL FOR SPEED!!!
# ======================================================
def ensure_indexes():
    with driver.session(database=DATABASE) as session:
        session.run("""
            CREATE INDEX IF NOT EXISTS FOR (s:spiRecord) ON (s.id)
        """)
        session.run("""
            CREATE INDEX IF NOT EXISTS FOR (t:Tag) ON (t.tagNumber)
        """)
    print("Indexes ensured.\n")


# ======================================================
# 2. FAST AURA‑OPTIMIZED CYPHER
# ======================================================
BATCH_QUERY = """
UNWIND $batch AS row
CREATE (s:spiRecord)
set s.siteName= "Gorgon"

WITH s, row,
     [k IN keys(row.props_create)
      WHERE row.props_create[k] IS NOT NULL
        AND row.props_create[k] = row.props_create[k]
        AND NOT toString(row.props_create[k]) IN ["", "-", "--", "---", "unset", "*"]] AS keepKeys

FOREACH (k IN keepKeys | SET s[k] = row.props_create[k])

WITH s, row
WHERE row.tagNumber IS NOT NULL
  AND row.tagNumber = row.tagNumber
  AND NOT toString(row.tagNumber) IN ["", "-", "--", "---", "unset", "*"]

MATCH (t:Tag {tagNumber: row.tagNumber})
MERGE (t)-[:PRESENT_IN]->(s);
"""


# ======================================================
# 3. Batch Upload Function (Super Fast)
# ======================================================


def upload_spi_fast(df_spi: pd.DataFrame, batch_size=5000):
    SENTINELS = {"", "-", "--", "---", "unset", "*"}

    print("Reading DF...")
    #df = pd.read_csv(spi_file, sep="\t", dtype=str).fillna("")
    df = df_spi.copy().fillna("")
    # Add ID column if missing
    if "id" not in df.columns:
        df.insert(0, "id", df.index.astype(str))

    # Vectorized strip
    df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

    # Rename project_status -> status
   # df = df.rename(columns={"project_status": "status"})

    # Only these columns exist (plus id). Keep them explicitly.
    PROP_COLS = [
        "category",
        "componentId",
        "componentName",
        "tagNumber",
        "discipline",
        "status",
        "projectNumber",
    ]

    print("Preparing rows...")
    rows = []
    for _, r in df[["id"] + PROP_COLS].iterrows():
        props_create = {}
        props_update = {}

        for col in PROP_COLS:
            val = r.get(col, "")

            # Skip NaN
            if val is None or (isinstance(val, float) and np.isnan(val)):
                continue

            # Normalize and filter sentinels/blanks
            if isinstance(val, str):
                v = val.strip()
                if v.lower() in SENTINELS:
                    continue
                val = v

            props_create[col] = val
            props_update[col] = val

        # Clean tag_number specifically for relationship logic
        tag_val = r.get("tagNumber", "")
        if tag_val is None or (isinstance(tag_val, float) and np.isnan(tag_val)):
            tag_val = ""
        else:
            t = str(tag_val).strip()
            tag_val = "" if t.lower() in SENTINELS else t

        rows.append({
            "id": str(r["id"]),
            "tagNumber": tag_val,
            "props_create": props_create,
            "props_update": props_update
        })

    print(f"Prepared {len(rows)} rows for upload.\n")

    # Write batches
    with driver.session(database=DATABASE) as session:
        batch = []
        batch_num = 0

        for i, row in enumerate(rows, start=1):
            batch.append(row)

            if len(batch) >= batch_size:
                batch_num += 1
                summary = session.run(BATCH_QUERY, batch=batch).consume().counters
                print(
                    f"Batch {batch_num} | "
                    f"{i}/{len(rows)} rows | "
                    f"Nodes={summary.nodes_created}, Rels={summary.relationships_created}"
                )
                batch.clear()

        # Final batch
        if batch:
            batch_num += 1
            summary = session.run(BATCH_QUERY, batch=batch).consume().counters
            print(
                f"Final Batch {batch_num} | "
                f"{len(rows)}/{len(rows)} rows | "
                f"Nodes={summary.nodes_created}, Rels={summary.relationships_created}"
            )

    print("\n🚀 Upload completed FAST.\n")


# ======================================================
# RUN
# ======================================================
ensure_indexes()
upload_spi_fast(df_spi)
driver.close()


Indexes ensured.

Reading DF...
Preparing rows...
Prepared 267367 rows for upload.

Batch 1 | 5000/267367 rows | Nodes=5000, Rels=16
Batch 2 | 10000/267367 rows | Nodes=5000, Rels=14
Batch 3 | 15000/267367 rows | Nodes=5000, Rels=16
Batch 4 | 20000/267367 rows | Nodes=5000, Rels=100
Batch 5 | 25000/267367 rows | Nodes=5000, Rels=118
Batch 6 | 30000/267367 rows | Nodes=5000, Rels=10
Batch 7 | 35000/267367 rows | Nodes=5000, Rels=8
Batch 8 | 40000/267367 rows | Nodes=5000, Rels=5
Batch 9 | 45000/267367 rows | Nodes=5000, Rels=16
Batch 10 | 50000/267367 rows | Nodes=5000, Rels=18
Batch 11 | 55000/267367 rows | Nodes=5000, Rels=86
Batch 12 | 60000/267367 rows | Nodes=5000, Rels=67
Batch 13 | 65000/267367 rows | Nodes=5000, Rels=0
Batch 14 | 70000/267367 rows | Nodes=5000, Rels=8
Batch 15 | 75000/267367 rows | Nodes=5000, Rels=1
Batch 16 | 80000/267367 rows | Nodes=5000, Rels=14
Batch 17 | 85000/267367 rows | Nodes=5000, Rels=6
Batch 18 | 90000/267367 rows | Nodes=5000, Rels=1
Batch 19 | 95

# SPI Wheatstone

### WSU

### Read Data

In [6]:
import pandas as pd
df_spi_u=pd.read_excel("SPI_WSU.xlsx")

### Columns to keep

In [8]:
columns_to_keep=['CAT', 'CMPNT_ID','CMPNT_NAME', 'TAG_NUMBER', 'DISCIPLINE','STATUS', 'PROJECT_NUMBER']
df_spi_u=df_spi_u[columns_to_keep]
rename_map={'CAT':'category', 'CMPNT_ID':'componentId','CMPNT_NAME':'componentName',  'TAG_NUMBER':'tagNumber', 'DISCIPLINE': 'discipline','STATUS':'status', 'PROJECT_NUMBER': 'projectNumber'}
df_spi_u=df_spi_u.rename(columns=rename_map)
#df_spi_u.to_csv('SPI_CVX_MTL.tsv', sep="\t", index=False)

###  Clean projectNumber column  and extract MocId

In [5]:
import pandas as pd
import re

def extract_mocs(value):
    """
    Extract and clean MOC numbers from the Project_no field.
    Returns a list of clean MOC values like 'MOC564805'.
    """

    if pd.isna(value):
        return []

    text = str(value)

    # Normalize separators
    text = text.replace("&", ",")
    text = text.replace(";", ",")
    text = text.replace("/", ",")
    text = text.replace(" and ", ",")
    text = text.replace("AND", ",")
    
    # Split multiple entries
    parts = re.split(r"[,\n]+", text)

    cleaned = []

    for p in parts:
        p = p.strip()

        # Find patterns with numbers and "MOC"
        match = re.search(r"(?:T?PMOC|TMOC|PMOC|MOC)[\s\-_]*([A-Za-z0-9]+)", p, re.IGNORECASE)

        if match:
            num = match.group(1).upper()
            cleaned.append("MOC" + num)
        else:
            # Catch patterns like "Rev 9 - MOC382271"
            match2 = re.search(r"MOC[\s\-_]*([A-Za-z0-9]+)", p, re.IGNORECASE)
            if match2:
                num = match2.group(1).upper()
                cleaned.append("MOC" + num)

    return cleaned


def clean_file(df_in: pd.DataFrame) -> pd.DataFrame:

    # Work on a copy to avoid mutating the caller's dataframe
    df = df_in.copy()

    # Create cleanMOC list column
    df["cleanMOC_list"] = df["projectNumber"].apply(extract_mocs)

    # Explode to multiple rows if more than one MOC
    df = df.explode("cleanMOC_list", ignore_index=True)

    # Rename final column
    df = df.rename(columns={"cleanMOC_list": "cleanMOC"})

    return df
df_spi_u=clean_file(df_spi_u)

### Load Moc node and Tag node from SPI

In [6]:
import pandas as pd
import numpy as np
from neo4j import GraphDatabase

# =========================
# Connection settings
# =========================


URI = os.getenv("NEO4J_URI")
USERNAME = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")
DATABASE = os.getenv("NEO4J_DATABASE")

SOR_NAME = "spi_wsu"
NULL_TOKENS = {"", "-", "--", "---", "unset", "*"}

# =========================
# Data prep (clean in Python)
# =========================

def prepare_spi_rows(df_in: pd.DataFrame) -> pd.DataFrame:
#def prepare_spi_rows(tsv_path: str):
    #df = pd.read_csv(tsv_path, sep="\t", dtype=str).fillna("")
    df=df_in.copy()
    # Trim whitespace
    df = df.applymap(lambda v: v.strip() if isinstance(v, str) else v)

    # Convert null-like tokens to None
    df = df.applymap(lambda v: None if (isinstance(v, str) and v in NULL_TOKENS) else v)

    # Convert real NaN to None
    df = df.where(pd.notna(df), None)

    rows = []
    for r in df.to_dict(orient="records"):
        mocId = r.get("cleanMOC")
        tag_num = r.get("tagNumber")

        # Only process rows that have both MOC id and tag_number
        if not mocId or not tag_num:
            continue

        # Exclude matching keys and source_file so t += row won't overwrite it
        props = {
            k: v for k, v in r.items()
            if k not in ["cleanMOC", "tagNumber", "sourceFile"] and v is not None
        }
        props["cleanMOC"] = mocId
        props["tagNumber"] = tag_num

        rows.append(props)

    print(f"Prepared {len(rows)} rows for upload.")
    return rows

# =========================
# Cypher (no MOC creation; Tag MERGE; append source_file; relate if MOC exists)
# =========================
UPSERT_CYPHER = """

UNWIND $rows AS row


MATCH (m:Moc {mocId: row.cleanMOC})
SET m.sourceFile =
  CASE
    WHEN m.sourceFile IS NULL THEN ['spi']
    WHEN 'spi' IN m.sourceFile THEN m.sourceFile
    ELSE m.sourceFile + ['spi']
  END

// Clean tag number
WITH row, m,
CASE
  WHEN row.tagNumber IS NOT NULL
       AND row.tagNumber = row.tagNumber              // exclude NaN
       AND NOT toString(row.tagNumber) IN ["", "-", "--", "---", "unset", "*"]
  THEN row.tagNumber
  ELSE NULL
END AS tagVal

// Create/match Tag ONLY when tagVal is valid
FOREACH (_ IN CASE WHEN tagVal IS NULL THEN [] ELSE [1] END |
  MERGE (t:Tag {tagNumber: tagVal})
  SET
    // only allowed properties on Tag
    t.discipline = coalesce(row.discipline, t.discipline),
    // normalize (note: backslash must be escaped as \\\\ in Python, \\ in Cypher)
    t.tagNorm   = replace(replace(replace(replace(toLower(tagVal), "-", ""), "\\\\", ""), " ", ""), "f", ""),
    t.tagTokens = [tok IN split(tagVal, "-") WHERE trim(tok) <> "" | toLower(trim(tok))],
    // append-only source_file (no overwrite, no duplicates)
    t.sourceFile =
      CASE
        WHEN t.sourceFile IS NULL THEN ['spi_wsu']
        WHEN 'spi_wsu' IN t.sourceFile THEN t.sourceFile
        ELSE t.sourceFile + ['spi_wsu']
      END, 
      t.siteName="WSU"
      
  MERGE (m)-[:ASSOCIATED_WITH_TAG]->(t)
)
"""





# =========================
# Run
# =========================
if __name__ == "__main__":
    

    rows = prepare_spi_rows(df_spi_u)

    driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
    try:
        with driver.session(database=DATABASE) as session:
            session.run(UPSERT_CYPHER, rows=rows, sor=SOR_NAME).consume()
        print("✅ Ingestion completed.")
    finally:
        driver.close()

C:\Users\rasma\AppData\Local\Temp\ipykernel_10616\3601536978.py:34: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda v: v.strip() if isinstance(v, str) else v)
C:\Users\rasma\AppData\Local\Temp\ipykernel_10616\3601536978.py:37: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda v: None if (isinstance(v, str) and v in NULL_TOKENS) else v)


Prepared 249 rows for upload.
✅ Ingestion completed.


### Load spiRecord node 

In [9]:
import pandas as pd
from neo4j import GraphDatabase
import numpy as np




import pandas as pd
from neo4j import GraphDatabase


URI = os.getenv("NEO4J_URI")
USERNAME = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")
DATABASE = os.getenv("NEO4J_DATABASE")

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))


# ======================================================
# 1. Ensure INDEXES — CRITICAL FOR SPEED!!!
# ======================================================
def ensure_indexes():
    with driver.session(database=DATABASE) as session:
        session.run("""
            CREATE INDEX IF NOT EXISTS FOR (s:spiRecord) ON (s.id)
        """)
        session.run("""
            CREATE INDEX IF NOT EXISTS FOR (t:Tag) ON (t.tagNumber)
        """)
    print("Indexes ensured.\n")


# ======================================================
# 2. FAST AURA‑OPTIMIZED CYPHER
# ======================================================
BATCH_QUERY = """
UNWIND $batch AS row
CREATE (s:spiRecord)
set s.siteName= "WSU"

WITH s, row,
     [k IN keys(row.props_create)
      WHERE row.props_create[k] IS NOT NULL
        AND row.props_create[k] = row.props_create[k]
        AND NOT toString(row.props_create[k]) IN ["", "-", "--", "---", "unset", "*"]] AS keepKeys

FOREACH (k IN keepKeys | SET s[k] = row.props_create[k])

WITH s, row
WHERE row.tagNumber IS NOT NULL
  AND row.tagNumber = row.tagNumber
  AND NOT toString(row.tagNumber) IN ["", "-", "--", "---", "unset", "*"]

MATCH (t:Tag {tagNumber: row.tagNumber})
MERGE (t)-[:PRESENT_IN]->(s);
"""


# ======================================================
# 3. Batch Upload Function (Super Fast)
# ======================================================


def upload_spi_fast(df_spi: pd.DataFrame, batch_size=5000):
    SENTINELS = {"", "-", "--", "---", "unset", "*"}

    print("Reading DF...")
    #df = pd.read_csv(spi_file, sep="\t", dtype=str).fillna("")
    df = df_spi.copy().fillna("")
    # Add ID column if missing
    if "id" not in df.columns:
        df.insert(0, "id", df.index.astype(str))

    # Vectorized strip
    df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

    # Rename project_status -> status
   # df = df.rename(columns={"project_status": "status"})

    # Only these columns exist (plus id). Keep them explicitly.
    PROP_COLS = [
        "category",
        "componentId",
        "componentName",
        "tagNumber",
        "discipline",
        "status",
        "projectNumber",
    ]

    print("Preparing rows...")
    rows = []
    for _, r in df[["id"] + PROP_COLS].iterrows():
        props_create = {}
        props_update = {}

        for col in PROP_COLS:
            val = r.get(col, "")

            # Skip NaN
            if val is None or (isinstance(val, float) and np.isnan(val)):
                continue

            # Normalize and filter sentinels/blanks
            if isinstance(val, str):
                v = val.strip()
                if v.lower() in SENTINELS:
                    continue
                val = v

            props_create[col] = val
            props_update[col] = val

        # Clean tag_number specifically for relationship logic
        tag_val = r.get("tagNumber", "")
        if tag_val is None or (isinstance(tag_val, float) and np.isnan(tag_val)):
            tag_val = ""
        else:
            t = str(tag_val).strip()
            tag_val = "" if t.lower() in SENTINELS else t

        rows.append({
            "id": str(r["id"]),
            "tagNumber": tag_val,
            "props_create": props_create,
            "props_update": props_update
        })

    print(f"Prepared {len(rows)} rows for upload.\n")

    # Write batches
    with driver.session(database=DATABASE) as session:
        batch = []
        batch_num = 0

        for i, row in enumerate(rows, start=1):
            batch.append(row)

            if len(batch) >= batch_size:
                batch_num += 1
                summary = session.run(BATCH_QUERY, batch=batch).consume().counters
                print(
                    f"Batch {batch_num} | "
                    f"{i}/{len(rows)} rows | "
                    f"Nodes={summary.nodes_created}, Rels={summary.relationships_created}"
                )
                batch.clear()

        # Final batch
        if batch:
            batch_num += 1
            summary = session.run(BATCH_QUERY, batch=batch).consume().counters
            print(
                f"Final Batch {batch_num} | "
                f"{len(rows)}/{len(rows)} rows | "
                f"Nodes={summary.nodes_created}, Rels={summary.relationships_created}"
            )

    print("\n🚀 Upload completed FAST.\n")


# ======================================================
# RUN
# ======================================================
ensure_indexes()
upload_spi_fast(df_spi_u)
driver.close()

Indexes ensured.

Reading DF...
Preparing rows...
Prepared 35071 rows for upload.

Batch 1 | 5000/35071 rows | Nodes=5000, Rels=84
Batch 2 | 10000/35071 rows | Nodes=5000, Rels=27
Batch 3 | 15000/35071 rows | Nodes=5000, Rels=23
Batch 4 | 20000/35071 rows | Nodes=5000, Rels=97
Batch 5 | 25000/35071 rows | Nodes=5000, Rels=0
Batch 6 | 30000/35071 rows | Nodes=5000, Rels=13
Batch 7 | 35000/35071 rows | Nodes=5000, Rels=50
Final Batch 8 | 35071/35071 rows | Nodes=71, Rels=0

🚀 Upload completed FAST.

